In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from matplotlib import animation
from matplotlib import cm
from matplotlib.collections import LineCollection
from IPython.display import HTML

import utilities.plot_settings
import pypopsyn.simulator.basics.constants as const
import pypopsyn.simulator.magneto_rotational_physics.period_derivative as pdv
import pypopsyn.simulator.stellar_dynamics.coordinate_conversions as coco
from pypopsyn.simulator.configuration import cfg

# Here you need to modify the path to adapt to your anaconda installation.
plt.rcParams['animation.ffmpeg_path'] = '/home/michele/miniconda3/envs/pop_syn/bin/ffmpeg'

polar_to_cartesian_vect = np.vectorize(coco.polar_to_cartesian)

# Dynamical evolution visualization

Visualize the evolution in time of the neutron stars' positions in galactocentric frame.

In [ ]:
# Read the .json data file containing the magneto-rotational time evolution information.
directory_path = "../examples/data/simulation_full_example/"

with open(f"{directory_path}dyn_evolution.json", "r") as read_file:
    dyn_evol_data = json.load(read_file)
    
star_idx = dyn_evol_data.keys()

In [ ]:
# Define a common time grid where to interpolate the data of different neutron stars.
t_grid = np.linspace(0., cfg["t_age_max"], 100)

x_t_interp = np.zeros( (len(star_idx),len(t_grid)) )
y_t_interp = np.zeros( (len(star_idx),len(t_grid)) )
z_t_interp = np.zeros( (len(star_idx),len(t_grid)) )

t_age = np.zeros(len(star_idx))

for i, key in enumerate(star_idx):
    t = dyn_evol_data[key]["t"]
    r_t = dyn_evol_data[key]["r(t)"]
    phi_t = dyn_evol_data[key]["phi(t)"]
    x_t, y_t = polar_to_cartesian_vect(r_t, phi_t)
    z_t = dyn_evol_data[key]["z(t)"]
    
    # Re-align the time arrays so that the evolution of each star starts at t_age_max - t_age and ends at t = t_age_max (now).
    t_shift = cfg["t_age_max"]-np.max(t)
    t_shifted = t + t_shift
    
    # Interpolate data on the common time grid.
    x_t_interp[i] = np.interp(t_grid, t_shifted, x_t)
    y_t_interp[i] = np.interp(t_grid, t_shifted, y_t)
    z_t_interp[i] = np.interp(t_grid, t_shifted, z_t)
    
    # Save the age in Myr
    t_age[i] = t[-1] / 1.e6

Animation of the dynamical evolution in the $xy$ plain.

In [ ]:
# Define the cmap for the colorbar.
t_age_norm = (t_age - min(t_age)) / (max(t_age) - min(t_age))
colors = cm.jet(t_age_norm)

norm = mpl.colors.Normalize(vmin=min(t_age), vmax=max(t_age))
cmap = mpl.cm.ScalarMappable(norm=norm, cmap=mpl.cm.jet)

fig, ax = plt.subplots(figsize=(11,8))

ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20., 20.)
ax.set_ylim(-20., 20.)

ax.plot(0.0, 8.3, marker="*", color="black", markersize=10)

# Initialize the various element of the plot for the animation. 
lines = []

for i in range(len(star_idx)):
    line_i, = ax.plot([],[],lw=2, color=colors[i], rasterized=True)    
    lines.append(line_i)
    
time_text = ax.text(0.3, 0.05, '', transform=ax.transAxes, fontsize=20)
        
x_data = [[] for i in range(len(star_idx))]
y_data = [[] for i in range(len(star_idx))]

# Initialization function for the lines.
def init():
    for line in lines:
        line.set_data([],[])
    return lines
    
# Animation function. This is called sequentially.
def animate(frame):
    time_text.set_text('time = %.0f kyr' % (t_grid[frame]/1000.))
    for lnum, line in enumerate(lines):
        x_data[lnum].append(x_t_interp[lnum,frame])
        y_data[lnum].append(y_t_interp[lnum,frame])
        line.set_data(x_data[lnum], y_data[lnum])

    return lines
    
axcb = fig.colorbar(cmap)
axcb.set_label(r'Age [Myr])')
plt.close()

anim = animation.FuncAnimation(fig, animate, frames=np.arange(0, len(t_grid)), interval=100, blit=False, repeat=False)

HTML(anim.to_html5_video())

Animation of the dynamical evolution in the $xz$ plain.

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))

ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20., 20.)
ax.set_ylim(-20., 20.)

ax.plot(0.0, 0.02, marker="*", color="black", markersize=10)

# Initialize the various element of the plot for the animation. 
lines = []

for i in range(len(star_idx)):
    line_i, = ax.plot([],[],lw=2, color=colors[i], rasterized=True)    
    lines.append(line_i)
    
time_text = ax.text(0.3, 0.05, '', transform=ax.transAxes, fontsize=20)
        
x_data = [[] for i in range(len(star_idx))]
y_data = [[] for i in range(len(star_idx))]

# Initialization function for the lines.
def init():
    for line in lines:
        line.set_data([],[])
    return lines
    
# Animation function. This is called sequentially.
def animate(frame):
    time_text.set_text('time = %.0f kyr' % (t_grid[frame]/1000.))
    for lnum, line in enumerate(lines):
        x_data[lnum].append(x_t_interp[lnum,frame])
        y_data[lnum].append(z_t_interp[lnum,frame])
        line.set_data(x_data[lnum], y_data[lnum])

    return lines
    
axcb = fig.colorbar(cmap)
axcb.set_label(r'Age [Myr])')
plt.close()

anim = animation.FuncAnimation(fig, animate, frames=np.arange(0, len(t_grid)), interval=100, blit=False, repeat=False)

HTML(anim.to_html5_video())

# Magneto-rotational evolution visualization

Visualize the evolution in time of the neutron stars in the $P \dot{P}$ dyagram.

In [ ]:
period_derivative_vect = np.vectorize(pdv.period_derivative)

def Pdot_tage(P:np.ndarray, tau_age:float) -> np.ndarray:
    '''
    Pdot as a function of period for varying characteristic age.
    Args:
        P (np.ndarray): Array of period values in [s].
        tau_age (float): characteristic age in [yr].
    Return:
        (np.ndarray): Values of period derivatives in [s/s].
    '''
    tau_age = tau_age * const.YR_TO_S
    Pdot = P / (2. * tau_age)
    return Pdot

In [ ]:
# Read the .json data file containing the magneto-rotational time evolution information.
directory_path = "../examples/data/simulation_maxwell_sigma265_h018/"

with open(f"{directory_path}magrot_evolution.json", "r") as read_file:
    magrot_evol_data = json.load(read_file)
    
star_idx = magrot_evol_data.keys()

In [ ]:
# Define a common time grid where to interpolate the data of different neutron stars.
t_grid = np.linspace(0., cfg["t_age_max"], 100)

P_t_interp = np.zeros( (len(star_idx),len(t_grid)) )
Pdot_t_interp = np.zeros( (len(star_idx),len(t_grid)) )
B_t_interp = np.zeros( (len(star_idx),len(t_grid)) )

logB_initial = np.zeros(len(star_idx))

for i, key in enumerate(star_idx):
    t = magrot_evol_data[key]["t"]
    B_t = magrot_evol_data[key]["B(t)"]
    chi_t = magrot_evol_data[key]["chi(t)"]
    P_t = magrot_evol_data[key]["P(t)"]
    
    Pdot_t = period_derivative_vect(B_t, chi_t, P_t) / const.YR_TO_S
    
    # Re-align the time arrays so that the evolution of each star starts at t_age_max - t_age and ends at t = t_age_max (now).
    t_shift = cfg["t_age_max"]-np.max(t)
    t_shifted = t + t_shift
    
    # Interpolate data on the common time grid.
    P_t_interp[i] = np.interp(t_grid, t_shifted, P_t)
    Pdot_t_interp[i] = np.interp(t_grid, t_shifted, Pdot_t)
    B_t_interp[i] = np.interp(t_grid, t_shifted, B_t)
    
    logB_initial[i] = np.log10(B_t[0])

In [ ]:
# Setup the P-Pdot plot.
B_grid = 10**(1.*(np.arange(10,18,2)))    # array of dipolar magnetic field values [gauss]
tau_age_grid = 10**(1.*(np.arange(2,14,4)))    # array of characteristic age values [yr]
P_grid = np.logspace(np.log10(2.e-3), np.log10(100),10)    # periods array [s]

fig, ax = plt.subplots(figsize=(12,8))

# Plot the lines of constant magnetic field and characteristic age.
for i in range(len(B_grid)):
    ax.plot(
        P_grid, 
        period_derivative_vect(B_grid[i], 0., P_grid) / const.YR_TO_S, 
        linestyle='-', 
        color='black',
        linewidth=1,
        zorder=1,
        rasterized=True
    )
    
for i in range(len(tau_age_grid)):
    ax.plot(
        P_grid, 
        Pdot_tage(P_grid,tau_age_grid[i]), 
        linestyle='--',
        color='black',
        linewidth=1,
        zorder=1,
        rasterized=True
    )

ax.text(3.e-3, 8e-18, '$10^{10}$ G', fontsize = 20, rotation=-25, color = 'black')
ax.text(3.e-3, 8e-14, '$10^{12}$ G', fontsize = 20, rotation=-25, color = 'black')
ax.text(1.5e-2, 1.5e-10, '$10^{14}$ G', fontsize = 20, rotation=-25, color = 'black')

ax.text(3.e-3, 8e-13, '$10^{2}$ yr', fontsize = 20, rotation=25, color = 'black')
ax.text(3.e-3, 8e-17, '$10^{6}$ yr', fontsize = 20, rotation=25, color = 'black')
ax.text(3., 8e-18, '$10^{10}$ yr', fontsize = 20, rotation=25, color = 'black')

ax.set_xlabel(r"Spin period [s]")
ax.set_ylabel(r"Spin period derivative [s/s]")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(2.e-3, 100)
ax.set_ylim(5.e-18, 1.e-9)

# Plot the trajectories of the stars in the P-Pdot diagram. 
# The varying color of the trajectories shows the evolution of the magnetic field strength.
for i in range(len(star_idx)):
    x = P_t_interp[i]
    y = Pdot_t_interp[i]
    z = np.log10(B_t_interp[i])

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    lc = LineCollection(segments, cmap=plt.get_cmap('jet'), norm=plt.Normalize(11, 15), rasterized=True)
    lc.set_array(z)
    lc.set_linewidth(2)

    plt.gca().add_collection(lc)

axcb = fig.colorbar(lc)
axcb.set_label(r'$\log_{10}(B_{\rm d}$ [Gauss])')

plt.show()


Animation of the magneto-rotational evolution in the $P \dot{P}$ dyagram.

In [ ]:
# Define the cmap for the colorbar.
logB_in_norm = (logB_initial - min(logB_initial)) / (max(logB_initial) - min(logB_initial))
colors = cm.jet(logB_in_norm)

norm = mpl.colors.Normalize(vmin=min(logB_initial), vmax=max(logB_initial))
cmap = mpl.cm.ScalarMappable(norm=norm, cmap=mpl.cm.jet)

# Setup the P-Pdot plot
fig, ax = plt.subplots(figsize=(12,8))

# plot the lines of constant magnetic field and characteristic age
for i in range(len(B_grid)):
    ax.plot(
        P_grid, 
        period_derivative_vect(B_grid[i], 0., P_grid) / const.YR_TO_S, 
        linestyle='-', 
        color='black',
        linewidth=1,
        zorder=1,
        rasterized=True
    )
    
for i in range(len(tau_age_grid)):
    ax.plot(
        P_grid, 
        Pdot_tage(P_grid,tau_age_grid[i]), 
        linestyle='--',
        color='black',
        linewidth=1,
        zorder=1,
        rasterized=True
    )

ax.text(3.e-3, 8e-18, '$10^{10}$ G', fontsize = 20, rotation=-25, color = 'black')
ax.text(3.e-3, 8e-14, '$10^{12}$ G', fontsize = 20, rotation=-25, color = 'black')
ax.text(1.5e-2, 1.5e-10, '$10^{14}$ G', fontsize = 20, rotation=-25, color = 'black')

ax.text(3.e-3, 8e-13, '$10^{2}$ yr', fontsize = 20, rotation=25, color = 'black')
ax.text(3.e-3, 8e-17, '$10^{6}$ yr', fontsize = 20, rotation=25, color = 'black')
ax.text(3., 8e-18, '$10^{10}$ yr', fontsize = 20, rotation=25, color = 'black')

ax.set_xlabel(r"Spin period [s]")
ax.set_ylabel(r"Spin period derivative [s/s]")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(2.e-3, 100)
ax.set_ylim(5.e-18, 1.e-9)

# Initialize the various element of the plot for the animation. 
lines = []

for i in range(len(star_idx)):
    line_i, = ax.plot([],[],lw=2, color=colors[i], rasterized=True)    
    lines.append(line_i)
    
time_text = ax.text(0.3, 0.05, '', transform=ax.transAxes, fontsize=20)
        
x_data = [[] for i in range(len(star_idx))]
y_data = [[] for i in range(len(star_idx))]

# Initialization function for the lines.
def init():
    for line in lines:
        line.set_data([],[])
    return lines
    
# Animation function. This is called sequentially.
def animate(frame):
    time_text.set_text('time = %.0f kyr' % (t_grid[frame]/1000.))
    for lnum, line in enumerate(lines):
        x_data[lnum].append(P_t_interp[lnum,frame])
        y_data[lnum].append(Pdot_t_interp[lnum,frame])
        line.set_data(x_data[lnum], y_data[lnum])

    return lines
    
axcb = fig.colorbar(cmap)
axcb.set_label(r'$\log_{10}({\rm initial} B_{\rm d}$ [Gauss])')
plt.close()

anim = animation.FuncAnimation(fig, animate, frames=np.arange(0, len(t_grid)), interval=100, blit=False, repeat=False)

HTML(anim.to_html5_video())